# Laboratorio 02 — Cómo está hecha la libreta

**Duración: 45 minutos.**

En el laboratorio anterior comprobaste que la tabla recuerda. Quedó una pregunta
abierta:

> La tabla no tiene una columna con las versiones. **¿Dónde está guardada esa
> memoria?**

Hoy la abrimos por dentro. Y no es curiosidad: de cómo está construida sale la
propiedad que a la DGT más le sirve, y que vas a ver con tus propios ojos al final
de la sesión —**dos motores distintos trabajando sobre la misma tabla, sin
coordinarse entre ellos**.

La plataforma de la DGT tiene Hive, Impala y Spark. Hoy cada motor tiene sus propias
reglas sobre las tablas y no siempre coinciden: una tabla que Hive lee bien, Impala la
lee distinto. Con Iceberg la definición es una sola.

---

### La libreta, por dentro

- Cada **página fechada** es un snapshot. Eso ya lo conoces.
- La libreta tiene un **índice**: unas hojas que dicen qué anotaciones pertenecen a
  qué página. En Iceberg eso son los **manifiestos**.
- Y la libreta cuelga de un **clavo en la pared**. El clavo no es la libreta: es lo
  que indica *cuál* libreta es la vigente. Eso es el **catálogo**.

Lo importante del clavo: **dos personas distintas que miran el mismo clavo leen la
misma libreta**. Ese es, exactamente, el motivo por el que dos motores ven la misma
tabla.

---

### Cómo se usa este notebook

Como los anteriores: `%%sql` ya viene en la primera línea, tú escribes la sentencia
debajo, una por celda, y ejecutas con **Shift + Enter**.

**Aviso de hoy, y ahórrate el primer tropiezo.** A partir del paso 2 vas a consultar
unas vistas del sistema —`.snapshots`, `.manifests`, `.files`—. Esas se nombran
**siempre con las tres partes**:

```
{DB}.{TABLA}.snapshots
 ^^^^^^^    ^^^^^^^^^^^^^^^^^^^^    ^^^^^^^^^
 tu espacio       tu tabla          la vista
```

**Aunque ya hayas hecho `USE`.** Si escribes solo `{TABLA}.snapshots`, SQL entiende
que `{TABLA}` es el nombre de un espacio de trabajo y te dice que no lo encuentra.
No es un error tuyo: es cómo se resuelven los nombres.


## Paso 0 — Preparar la tabla

Este laboratorio no depende de ninguno anterior. Empezamos por ponerte en tu
espacio de trabajo.

In [ ]:
%%sql
-- Celda 0.1



Partimos de cero, por si ya lo habías corrido antes.

In [ ]:
%%sql
-- Celda 0.2



La tabla del laboratorio. Tres columnas y `USING iceberg`, como siempre.

In [ ]:
%%sql
-- Celda 0.3



Un primer lote de tres contribuyentes.

In [ ]:
%%sql
-- Celda 0.4



Y un segundo lote, que llegó después. Dos cargas separadas, como pasa de verdad
con las incorporaciones al padrón: **eso nos va a dar dos páginas para mirar**.

In [ ]:
%%sql
-- Celda 0.5



Comprobemos qué quedó.

In [ ]:
%%sql
-- Celda 0.6



## Paso 1 — La tabla se describe a sí misma

Cuatro filas, cargadas en dos veces.

Ahora la primera pregunta: **¿qué es exactamente esta tabla?** No en el sentido de
qué datos tiene, sino de qué es como objeto. Pidámosle que se describa.

In [ ]:
%%sql
-- Celda 1.1



Mira lo que devolvió, porque hay más de lo que parece:

- Las **columnas y sus tipos**, que tú declaraste.
- Un `Provider` que dice **iceberg**.
- Una `Location`: una ruta del almacenamiento. **La tabla vive en una carpeta**, y
  esa carpeta tiene nombre.

Esa definición no está en el motor. Está guardada, y cualquiera que sepa dónde
buscarla la lee igual. Pidamos ahora las propiedades.

In [ ]:
%%sql
-- Celda 1.2



## Paso 2 — Las páginas y el índice

`format = iceberg/parquet`: los datos están en archivos Parquet y la tabla es
Iceberg. Y `current-snapshot-id`: **cuál página manda ahora mismo**. Ese es el clavo.

Vamos a las páginas. **Fíjate en el nombre a tres partes**, que es lo que avisamos al
principio: tu espacio, tu tabla, y después la vista.

In [ ]:
%%sql
-- Celda 2.1



Dos páginas, una por cada carga. El `current-snapshot-id` que viste en el paso
anterior es el de la segunda: la vigente.

El historial cuenta lo mismo desde otro ángulo: qué página fue la vigente y desde
cuándo.

In [ ]:
%%sql
-- Celda 2.2



Ahora el índice de la libreta. Un **manifiesto** es la hoja que dice qué archivos
de datos pertenecen a una página, y cuántos.

Esto es lo que le permite a Iceberg saber qué leer **sin salir a mirar la carpeta**:
no lista el directorio, lee el índice.

In [ ]:
%%sql
-- Celda 2.3



## Paso 3 — Dónde viven los datos

Un manifiesto por carga. Fíjate en `added_data_files_count`: cada uno agregó sus
archivos.

Y por fin, los archivos mismos. Esta es la vista que responde la pregunta que un
administrador se hace todos los días: **¿cuántos archivos tiene mi tabla y cuánto
pesan?**

In [ ]:
%%sql
-- Celda 3.1



Ahí está el fondo del asunto: **la tabla es un conjunto de archivos Parquet más
unos índices que dicen cuáles cuentan**. Nada más. No hay un servidor guardando el
estado, no hay una base de datos escondida.

Pidamos el resumen, que es lo que uno mira en producción.

In [ ]:
%%sql
-- Celda 3.2



## Paso 4 — El mismo dato desde otro motor

Cuatro filas repartidas en unos pocos archivos chicos. Anota ese número de
archivos: en el laboratorio 08 vas a ver por qué importa.

---

**Ahora el momento del laboratorio.** Todo lo que viste —la definición, las páginas,
el índice, los archivos— está **guardado**, no vive dentro de Spark. Así que otro
motor que mire el mismo clavo debería ver exactamente lo mismo.

Vamos a comprobarlo. **Abre Hue en otra pestaña** —tu instructor te da la
dirección—, entra con tu usuario y abre el editor **SparkSQL (Iceberg)**.

Ahí, en Hue y no aquí, ejecuta la consulta que te indique el instructor sobre tu
tabla. **Vas a ver las mismas cuatro filas.**

Después, también en Hue, agrega una fila más. Y cuando lo hayas hecho, vuelve a este
notebook.

In [ ]:
%%sql
-- Celda 4.1



Puede que ahí veas **cuatro filas todavía**, y no cinco. No es que el dato no haya
llegado: es que **este motor te está mostrando lo que ya tenía en memoria**.

Para no ir a preguntar por la definición de la tabla en cada consulta, el motor se
guarda una copia de lo que leyó. Esa copia se vence sola al cabo de unos segundos,
pero no vale la pena esperarla: se le pide que refresque y listo.

Esto **no es coordinarse con el otro motor.** El dato ya estaba escrito desde el
momento en que Hue lo confirmó; el único que estaba atrasado era el cuaderno que tu
motor tenía en el escritorio.

In [ ]:
%%sql
-- Celda 4.2



Y ahora sí, la misma consulta de recién.

In [ ]:
%%sql
-- Celda 4.3



**Cinco filas.** La que escribiste en Hue está aquí, y no hiciste nada para
traerla: no exportaste, no copiaste, no sincronizaste nada entre los dos motores.

Y no es que Spark haya ido a preguntarle a Hue. Miremos las páginas otra vez.

In [ ]:
%%sql
-- Celda 4.4



---

## Listo

**Tres páginas.** Las dos de tus cargas, y una tercera: la que escribió Hue. Para la
tabla, la escritura de Hue es una página más — igual que las tuyas.

### Lo que te llevas

La tabla Iceberg es **tres cosas guardadas**, ninguna de ellas dentro de un motor:

| En la libreta | En Iceberg | Qué contiene |
|---|---|---|
| La página fechada | **Snapshot** | Qué estado de la tabla es ese |
| El índice | **Manifiesto** | Qué archivos pertenecen a esa página |
| Las anotaciones | **Archivos de datos** | Los datos, en Parquet |
| El clavo en la pared | **Catálogo** | Cuál página manda ahora |

Y de ahí sale, sin nada más, lo que viste en el paso 4: **no hay dos libretas, hay
una sola colgada del mismo clavo.** Spark y el motor de Hue no se hablan entre ellos
y no lo necesitan; los dos miran el mismo clavo y leen la misma libreta.

Para la DGT eso significa que Hive, Impala y Spark dejan de tener cada uno su propia
idea de qué es una tabla.

### La pregunta del laboratorio siguiente

Tu tabla creció ordenada porque cargaste datos limpios. Pero el feed de recepción de
la DGT no es limpio: **el mismo documento llega dos veces**, porque un contribuyente
reenvía lo que cree que no llegó.

Si haces `INSERT` de un folio que ya está, ahora sabes exactamente qué va a pasar: se
va a escribir una página nueva con el documento repetido, y la tabla va a tener dos.

**¿Cómo se carga entonces, sin duplicar?** Eso es el **Laboratorio 03 — El folio que
llegó dos veces**.
